# Chapter 3: Planning Algorithms — Value Iteration and Linear Programming

**S&DS 685: Reinforcement Learning** | Yale University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZhuoranYang/sds685-notes/blob/main/notebooks/03-planning-algorithms.ipynb)

This notebook accompanies [Chapter 3: MDP Planning](../lectures/03-mdp-planning.qmd). We implement the core planning algorithms on the **Frozen Lake** environment:

1. **Value Iteration** (V-function version)
2. **Value Iteration** (Q-function version)
3. **Linear Programming**

All three methods compute the same optimal value function $V^*$ — we verify this numerically.

## Setup

Run this cell first to install dependencies (only needed on Colab).

In [ ]:
# Colab setup — skip if running locally
import importlib
if importlib.util.find_spec('gymnasium') is None:
    !pip install -q gymnasium[toy-text] seaborn

In [ ]:
import gymnasium as gym
from gymnasium.envs.toy_text.frozen_lake import generate_random_map
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import optimize

## 1. The Frozen Lake Environment

Frozen Lake is a grid-world MDP where an agent must navigate from **S** (start) to **G** (goal) on a frozen lake, avoiding **H** (holes). The ice is slippery: each action has a $\tfrac{1}{3}$ chance of moving in the intended direction and $\tfrac{1}{3}$ for each perpendicular direction.

This is a finite MDP with:
- $|\mathcal{S}| = 64$ states (8×8 grid)
- $|\mathcal{A}| = 4$ actions (left, down, right, up)
- Deterministic rewards: $+1$ for reaching the goal, $0$ otherwise
- Stochastic transitions (when `is_slippery=True`)

In [ ]:
# Create the 8x8 Frozen Lake environment
size = 8
env = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True)

n_states = env.observation_space.n
n_actions = env.action_space.n
action_names = ['LEFT', 'DOWN', 'RIGHT', 'UP']

print(f"States: {n_states}, Actions: {n_actions}")
print(f"\nMap:")
for row in env.unwrapped.desc:
    print(''.join(c.decode() for c in row))

### Extracting the MDP model $P(s'|s,a)$ and $R(s,a)$

Gymnasium exposes the full transition model via `env.unwrapped.P[state][action]`, which returns a list of `(probability, next_state, reward, done)` tuples. We extract these into numpy arrays.

In [ ]:
# Extract transition probabilities P[s,a,s'] and expected rewards R[s,a]
P = np.zeros((n_states, n_actions, n_states))
R = np.zeros((n_states, n_actions))

for s in range(n_states):
    for a in range(n_actions):
        for prob, s_next, reward, done in env.unwrapped.P[s][a]:
            P[s, a, s_next] += prob
            R[s, a] += prob * reward

# Show transition probabilities from the start state
print("Transitions from state 0 (top-left corner):")
for a in range(n_actions):
    nonzero = np.nonzero(P[0, a])[0]
    probs = P[0, a, nonzero]
    print(f"  {action_names[a]:>5}: " + ", ".join(f"s'={s} (p={p:.2f})" for s, p in zip(nonzero, probs)))

## 2. Value Iteration (V-function)

Value iteration applies the Bellman optimality operator repeatedly:

$$V_{k+1}(s) = \max_a \left\{ R(s,a) + \gamma \sum_{s'} P(s'|s,a) \, V_k(s') \right\}$$

By the contraction theorem, $V_k \to V^*$ geometrically.

In [ ]:
def value_iteration(env, gamma=0.99, tol=1e-10, max_iter=100000):
    """Compute V* using value iteration."""
    n_s = env.observation_space.n
    n_a = env.action_space.n
    V = np.zeros(n_s)

    for k in range(max_iter):
        V_old = V.copy()
        for s in range(n_s):
            Q_values = np.zeros(n_a)
            for a in range(n_a):
                for prob, s_next, reward, _ in env.unwrapped.P[s][a]:
                    Q_values[a] += prob * (reward + gamma * V_old[s_next])
            V[s] = np.max(Q_values)

        if np.max(np.abs(V - V_old)) < tol:
            print(f"Value iteration converged at iteration {k+1}")
            return V

    print(f"Value iteration did not converge in {max_iter} iterations")
    return V


def extract_greedy_policy(env, V, gamma=0.99):
    """Extract the greedy policy from a value function."""
    n_s = env.observation_space.n
    n_a = env.action_space.n
    policy = np.zeros(n_s, dtype=int)

    for s in range(n_s):
        Q_values = np.zeros(n_a)
        for a in range(n_a):
            for prob, s_next, reward, _ in env.unwrapped.P[s][a]:
                Q_values[a] += prob * (reward + gamma * V[s_next])
        policy[s] = np.argmax(Q_values)

    return policy

In [ ]:
V_vi = value_iteration(env, gamma=0.99)
pi_vi = extract_greedy_policy(env, V_vi)

print("\nV* (reshaped as 8x8 grid):")
print(np.round(V_vi.reshape(size, size), 3))

## 3. Value Iteration (Q-function)

Instead of maintaining $V(s)$, we can directly iterate on $Q(s,a)$:

$$Q_{k+1}(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \max_{a'} Q_k(s',a')$$

The optimal policy is then $\pi^*(s) = \arg\max_a Q^*(s,a)$.

In [ ]:
def q_value_iteration(env, gamma=0.99, tol=1e-10, max_iter=100000):
    """Compute Q* using Q-value iteration."""
    n_s = env.observation_space.n
    n_a = env.action_space.n
    Q = np.zeros((n_s, n_a))

    for k in range(max_iter):
        Q_old = Q.copy()
        for s in range(n_s):
            for a in range(n_a):
                val = 0
                for prob, s_next, reward, _ in env.unwrapped.P[s][a]:
                    val += prob * (reward + gamma * np.max(Q_old[s_next]))
                Q[s, a] = val

        if np.max(np.abs(Q - Q_old)) < tol:
            print(f"Q-value iteration converged at iteration {k+1}")
            return Q

    print(f"Q-value iteration did not converge in {max_iter} iterations")
    return Q

In [ ]:
Q_star = q_value_iteration(env, gamma=0.99)
V_qvi = np.max(Q_star, axis=1)  # V*(s) = max_a Q*(s,a)

## 4. Linear Programming

The Bellman optimality equation can be cast as a linear program:

$$\min_{V} \sum_s V(s) \quad \text{s.t.} \quad V(s) \geq R(s,a) + \gamma \sum_{s'} P(s'|s,a) V(s'), \quad \forall \, s, a.$$

At the optimum, the constraints are tight for the optimal action at each state.

In [ ]:
def solve_mdp_lp(env, gamma=0.99):
    """Solve for V* using linear programming."""
    n_s = env.observation_space.n
    n_a = env.action_space.n

    # Extract P and R
    P = np.zeros((n_s, n_a, n_s))
    R_sa = np.zeros((n_s, n_a))
    for s in range(n_s):
        for a in range(n_a):
            for prob, s_next, reward, _ in env.unwrapped.P[s][a]:
                P[s, a, s_next] += prob
                R_sa[s, a] += prob * reward

    # Constraint: V(s) - gamma * sum_s' P(s'|s,a) V(s') >= R(s,a)
    # In matrix form: A @ V >= b
    n_constraints = n_s * n_a
    A = np.zeros((n_constraints, n_s))
    b = np.zeros(n_constraints)

    idx = 0
    for s in range(n_s):
        for a in range(n_a):
            A[idx, s] = 1.0
            A[idx, :] -= gamma * P[s, a, :]
            b[idx] = R_sa[s, a]
            idx += 1

    # Objective: minimize sum V(s)
    c = np.ones(n_s)

    # scipy convention: A_ub @ x <= b_ub, so we negate
    result = optimize.linprog(c, A_ub=-A, b_ub=-b, method='highs')

    if not result.success:
        raise ValueError(f"LP failed: {result.message}")

    return result.x

In [ ]:
V_lp = solve_mdp_lp(env, gamma=0.99)
print("LP solved successfully.")

## 5. Verification: All Three Methods Agree

The fundamental theorem guarantees a unique $V^*$. Let's verify.

In [ ]:
print("Maximum absolute differences between V* computed by each method:")
print(f"  VI  vs Q-VI: {np.max(np.abs(V_vi - V_qvi)):.2e}")
print(f"  VI  vs LP:   {np.max(np.abs(V_vi - V_lp)):.2e}")
print(f"  Q-VI vs LP:  {np.max(np.abs(V_qvi - V_lp)):.2e}")

## 6. Visualizing the Optimal Policy

In [ ]:
def plot_value_and_policy(V, policy, desc, size=8):
    """Plot the value function heatmap and optimal policy arrows."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    arrows = {0: '\u2190', 1: '\u2193', 2: '\u2192', 3: '\u2191'}  # ← ↓ → ↑
    cell_colors = {'S': '#7B97AD', 'F': '#D4C9B8', 'H': '#C47A6A', 'G': '#B8995E'}

    # --- Value function heatmap ---
    V_grid = V.reshape(size, size)
    ax = axes[0]
    sns.heatmap(V_grid, annot=True, fmt='.3f', cmap='YlOrBr', ax=ax,
                cbar_kws={'label': '$V^*(s)$'}, linewidths=0.5)
    ax.set_title('Optimal Value Function $V^*$', fontsize=14)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')

    # --- Policy grid ---
    ax = axes[1]
    for i in range(size + 1):
        ax.axhline(i, color='#3D3530', linewidth=0.5)
        ax.axvline(i, color='#3D3530', linewidth=0.5)
    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')

    for i in range(size):
        for j in range(size):
            cell_type = desc[i][j].decode()
            rect = plt.Rectangle((j, i), 1, 1, facecolor=cell_colors.get(cell_type, '#F9F6F0'),
                                 alpha=0.4)
            ax.add_patch(rect)
            ax.text(j + 0.15, i + 0.25, cell_type, fontsize=9, color='#7B7067')
            if cell_type not in ['H', 'G']:
                ax.text(j + 0.5, i + 0.55, arrows[policy[i * size + j]],
                        fontsize=18, ha='center', va='center', color='#3D3530')

    ax.set_title('Optimal Policy $\\pi^*$', fontsize=14)
    plt.tight_layout()
    plt.show()


plot_value_and_policy(V_vi, pi_vi, env.unwrapped.desc, size)

## 7. Simulating the Optimal Policy

In [ ]:
def simulate_and_plot(env, policy, desc, size=8, max_steps=200):
    """Simulate a trajectory and plot it on the grid."""
    state, _ = env.reset()
    trajectory = [state]
    total_reward = 0

    for _ in range(max_steps):
        action = int(policy[state])
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        trajectory.append(state)
        if terminated or truncated:
            break

    # Plot
    fig, ax = plt.subplots(figsize=(8, 8))
    cell_colors = {'S': '#7B97AD', 'F': '#D4C9B8', 'H': '#C47A6A', 'G': '#B8995E'}

    for i in range(size + 1):
        ax.axhline(i, color='#3D3530', linewidth=0.5)
        ax.axvline(i, color='#3D3530', linewidth=0.5)

    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')

    for i in range(size):
        for j in range(size):
            cell_type = desc[i][j].decode()
            rect = plt.Rectangle((j, i), 1, 1, facecolor=cell_colors.get(cell_type, '#F9F6F0'),
                                 alpha=0.3)
            ax.add_patch(rect)
            ax.text(j + 0.12, i + 0.25, cell_type, fontsize=9, color='#7B7067')

    cmap = plt.cm.Reds
    n_steps = len(trajectory)
    for step, s in enumerate(trajectory):
        r, c = divmod(s, size)
        color = cmap(step / max(1, n_steps - 1))
        circle = plt.Circle((c + 0.5, r + 0.5), 0.3, color=color, alpha=0.7)
        ax.add_patch(circle)
        ax.text(c + 0.5, r + 0.5, str(step), ha='center', va='center',
                color='white', fontweight='bold', fontsize=7)

    ax.set_title(f'Trajectory ({n_steps} steps, reward={total_reward:.0f})', fontsize=14)
    plt.tight_layout()
    plt.show()
    return trajectory, total_reward


traj, rew = simulate_and_plot(env, pi_vi, env.unwrapped.desc, size)

## 8. Deterministic vs Stochastic Transitions

Setting `is_slippery=False` makes the environment deterministic. The optimal policy becomes a shortest path.

In [ ]:
env_det = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=False)

V_det = value_iteration(env_det, gamma=0.99)
pi_det = extract_greedy_policy(env_det, V_det)

plot_value_and_policy(V_det, pi_det, env_det.unwrapped.desc, size)
traj_det, rew_det = simulate_and_plot(env_det, pi_det, env_det.unwrapped.desc, size)

env_det.close()

In [ ]:
env.close()